# Train classification model

Imports

In [1]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F
from ray.train import Checkpoint, get_checkpoint
from ray import tune
from tqdm.auto import tqdm
# Absolute path to your project's src folder
src_path = os.path.abspath(os.path.join("..", "src"))

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(sys.path)

from data import get_data_loaders, show_image
from models.classification_model import ClassificationModel
from parameter_optimization import optimize_parameters, get_best_config
import config

['/Users/bevanslabbert/Documents/GitHub/pid-radast/src', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/thirdparty_files', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python313.zip', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13', '/opt/homebrew/Cellar/python@3.13/3.13.5/Frameworks/Python.framework/Versions/3.13/lib/python3.13/lib-dynload', '', '/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages', '/var/folders/xl/d7wznkt15fq8wspgq66fyc9m0000gn/T/tmprkt5twkq']


Data Initialization

In [2]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((150, 150)),  # resize generated image
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
])

trainloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Resize((150, 150)),  # resize generated image
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])
    # transform_train) TODO: this is the correct transformer for data augmentation, not working atm

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified
torch.Size([2, 1, 150, 150])
[tensor([[[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -0.9765, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -0.9765, -0.9765,  ..., -1.0000, -1.0000, -1.0000]]],


        [[[-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          ...,
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
          [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.000

In [3]:
import torch.nn.functional as F

def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    device = next(model.parameters()).device

    images = images.clone().detach().to(device).float()
    labels = labels.clone().detach().to(device)
    ori_images = images.clone().detach()

    for _ in range(iters):
        images.requires_grad = True
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        model.zero_grad()
        loss.backward()
        grad = images.grad.sign()

        images = images + alpha * grad
        delta = torch.clamp(images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + delta, min=0, max=1).detach()

    return images


Parameter optimization

In [4]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, parent_dir)
model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']

for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'epoch {epoch+1}')
    running_loss = 0.0
    for data in tqdm(trainloader, 0):
        # get the inputs
        images, labels = data

        # forward on clean and adversarial
        outputs_clean = model(images)

        # combined loss
        loss_clean = F.cross_entropy(outputs_clean, labels)
        loss = loss_clean

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"epoch {epoch} | loss: {running_loss/len(trainloader):.4f}")

print('finished training')


2025-09-05 11:31:11,072	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 1 trial(s):
- train__5efc7_00029: FileNotFoundError('Could not fetch metrics for train__5efc7_00029: both result.json and progress.csv were not found at /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel/train__5efc7_00029_29_batch_size=8,lr=0.0001,optimizer_class=ref_ph_626dc8cf_2025-08-02_15-41-19')


Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel
epoch 1


  0%|          | 0/347 [00:01<?, ?it/s]

epoch 0 | loss: 0.7332
epoch 2


  0%|          | 0/347 [00:02<?, ?it/s]

...

epoch 7 | loss: 0.1605
epoch 9


  0%|          | 0/347 [00:01<?, ?it/s]

epoch 8 | loss: 0.1043
epoch 10


  0%|          | 0/347 [00:02<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get prediction for classification model
output = model(images)
_, predicted = torch.max(output, 1)

# get accuracy
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 0 %


### Adversarially attacked classifier training

In [ ]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, parent_dir)
adv_model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](adv_model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']

for epoch in range(config.ROBUST_CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    running_loss = 0.0
    for data in tqdm(trainloader, 0):
        # get the inputs
        images, labels = data

        # clean images
        outputs = adv_model(images)
        loss_clean = criterion(outputs, labels)

        # adversarially attacked images
        adv_images = pgd_attack(model, images, labels)
        outputs_adv = adv_model(adv_images)
        loss_adv = criterion(outputs_adv, labels)

        # combine losses
        running_loss += 0.5 * loss_adv.item() + 0.5 * loss_clean.item()

        optimizer.zero_grad()
        loss_adv.backward()
        optimizer.step()


    print(f"Epoch {epoch + 1} | Loss: {running_loss/len(trainloader):.4f}")

print('Finished Training')

2025-08-16 10:21:52,503	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 1 trial(s):
- train__5efc7_00029: FileNotFoundError('Could not fetch metrics for train__5efc7_00029: both result.json and progress.csv were not found at /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel/train__5efc7_00029_29_batch_size=8,lr=0.0001,optimizer_class=ref_ph_626dc8cf_2025-08-02_15-41-19')


Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel


  0%|          | 0/347 [00:02<?, ?it/s]

Epoch 1 | Loss: 0.8119


  0%|          | 0/347 [00:02<?, ?it/s]

...

Epoch 8 | Loss: 0.3071


  0%|          | 0/347 [00:02<?, ?it/s]

Epoch 9 | Loss: 0.3544


  0%|          | 0/347 [00:02<?, ?it/s]

Epoch 10 | Loss: 0.6843
Finished Training


In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get prediction for classification model
output = adv_model(images)
_, predicted = torch.max(output, 1)

# get accuracy
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = adv_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the robust classifier on the 88 test images: %d %%' % (100 * correct / total))

NameError: name 'testloader' is not defined